# scicp — Fine-tune MiniLM on Scripture (Kaggle)

Fine-tunes `all-MiniLM-L6-v2` on **~397k** training pairs from 8 English sources.

**Setup:**
1. Create a Kaggle Dataset called `scicp-training` and upload `training-pairs.json`
2. In notebook settings: **Accelerator → GPU T4 ×2** (or P100)
3. Add the dataset: **+ Add Data → Your Datasets → scicp-training**
4. Run all cells — ~15–20 min on T4
5. Download `scripture-minilm.zip` from the Output tab

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q sentence-transformers datasets accelerate

In [ ]:
# ── 2. Check GPU ─────────────────────────────────────────────────────────────
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"GPU: {gpu}  VRAM: {vram} GB")
    # Auto-select batch size — L6 is smaller, fits more per batch
    if vram >= 40:    MICRO_BATCH, GRAD_ACCUM = 512, 1   # A100
    elif vram >= 22:  MICRO_BATCH, GRAD_ACCUM = 256, 1   # L4/A10
    else:             MICRO_BATCH, GRAD_ACCUM = 128, 2   # T4 (15.6 GB)
    print(f"→ micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM}, effective_batch={MICRO_BATCH * GRAD_ACCUM}")
else:
    MICRO_BATCH, GRAD_ACCUM = 32, 8
    print("No GPU — training will be very slow")

In [ ]:
# ── 3. Locate training data ─────────────────────────────────────────────────
import os, glob, time

# Show what Kaggle mounted so we can debug path issues
print("/kaggle/input contents:")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(" ", os.path.join(root, f))

# Recursive search — handles both flat and nested dataset layouts
candidates = glob.glob("/kaggle/input/**/training-pairs.json", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "training-pairs.json not found under /kaggle/input/.\n"
        "Make sure you added the dataset via + Add Data → Your Datasets."
    )
LOCAL_PATH = candidates[0]
print(f"\nUsing: {LOCAL_PATH}  ({os.path.getsize(LOCAL_PATH)//1024//1024} MB)")

In [ ]:
# ── 4. Load + prepare dataset ────────────────────────────────────────────────
import json, random
from datasets import Dataset

t0 = time.time()
with open(LOCAL_PATH) as f:
    pairs = json.load(f)
print(f"Loaded {len(pairs):,} pairs in {time.time()-t0:.1f}s")

random.seed(42)
random.shuffle(pairs)

# 97/3 split — with 676k pairs, 3% validation (~20k) is more than enough
split = int(len(pairs) * 0.97)
train_ds = Dataset.from_dict({
    "anchor":   [p["anchor"]   for p in pairs[:split]],
    "positive": [p["positive"] for p in pairs[:split]],
})
val_ds = Dataset.from_dict({
    "anchor":   [p["anchor"]   for p in pairs[split:]],
    "positive": [p["positive"] for p in pairs[split:]],
})
del pairs  # free ~400 MB RAM

print(f"train={len(train_ds):,}  val={len(val_ds):,}")
print("Sample:", train_ds[0])

In [ ]:
# ── 5. Fine-tune ─────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

# English-only model: 6 layers, 384 dims, fast inference
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OUT_DIR    = "/kaggle/working/scripture-minilm"
EPOCHS     = 2       # 346k × 2 = 692k samples; plenty for convergence

effective_batch = MICRO_BATCH * GRAD_ACCUM  # 256 on T4
total_steps     = (len(train_ds) // effective_batch) * EPOCHS
warmup_steps    = total_steps // 20  # 5% warmup

print(f"Effective batch: {effective_batch}")
print(f"Steps/epoch: {len(train_ds) // effective_batch:,}")
print(f"Total steps: {total_steps:,}  Warmup: {warmup_steps}")
print(f"In-batch negatives per sample: {effective_batch - 1}")

model = SentenceTransformer(BASE_MODEL)
loss  = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    per_device_eval_batch_size=MICRO_BATCH * 2,  # no gradients = 2x fits
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup_steps,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
)

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"\n✅ Done in {elapsed:.1f} min  ({len(train_ds) * EPOCHS / elapsed:.0f} pairs/min)")

In [ ]:
# ── 6. Save model for download ──────────────────────────────────────────────
import os, shutil

OUT_DIR = "/kaggle/working/scripture-minilm"

# Save final model
model.save(OUT_DIR)
print("Model files:", [f for f in os.listdir(OUT_DIR) if not f.startswith("checkpoint")])

# Create zip for easy download from Kaggle Output tab
ZIP_PATH = "/kaggle/working/scripture-minilm.zip"
shutil.make_archive("/kaggle/working/scripture-minilm", "zip", OUT_DIR)
size_mb = os.path.getsize(ZIP_PATH) // 1024 // 1024
print(f"\n✅ scripture-minilm.zip ({size_mb} MB) ready in Output tab")
print("Download it and extract to: resources/models/scripture-minilm/")

## After training

Download `scripture-minilm.zip` from the **Output** tab (right side panel).

On your local machine:

```bash
# 1. Extract the zip to:
#    resources/models/scripture-minilm/

# 2. Re-encode all 41k verses with the fine-tuned model (~3 min)
python3 scripts/rebake-embeddings.py

# 3. Re-whiten embeddings
node scripts/prebake-whitening.js

# 4. Rebuild cluster labels
node scripts/prebake-cluster-labels.js

# 5. Rebuild kNN graph
node scripts/prebake-knn.js

# 6. Rebuild spectral embeddings
node scripts/prebake-spectral.js

# 7. Restart the server
npm run dev
```